### Basic Rule based Scoring

In [65]:
import pandas as pd
import numpy as np
import re

In [66]:
file_path="AutoEIT Sample Transcriptions for Scoring.xlsx"
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names
print(sheet_names)

['Info', '38001-1A', '38002-2A', '38004-2A', '38006-2A']


In [72]:
df=pd.read_excel(xls,sheet_name=sheet_names[1])
df.head()

,Sentence,Stimulus,Transcription Rater 1,Score
0,1,Quiero cortarme el pelo (7),Quiero cortarme el pelo,NaN
1,2,El libro está en la mesa (7),El libro está en la mesa,NaN
2,3,El carro lo tiene Pedro (8),El carro lo tiene Pedro,NaN
3,4,El se ducha cada mañana (9),El se ducha cada mañana,NaN
4,5,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?,NaN


In [68]:
def preprocess(text):
    if pd.isna(text):
        return "";
    text=text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text=re.sub(r'[^\w\s]','',text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [69]:
df["clean_stimulus"]=df["Stimulus"].apply(preprocess)
df["clean_original"]=df["Transcription Rater 1"].apply(preprocess)
df.head()

,Sentence,Stimulus,Transcription Rater 1,Score,clean_stimulus,clean_original
0,1,Quiero cortarme el pelo (7),Quiero cortarme mi pelo,NaN,quiero cortarme el pelo 7,quiero cortarme mi pelo
1,2,El libro está en la mesa (7),El libro [pause] está en la mesa,NaN,el libro está en la mesa 7,el libro está en la mesa
2,3,El carro lo tiene Pedro (8),E-[gibberish] perro,NaN,el carro lo tiene pedro 8,e perro
3,4,El se ducha cada mañana (9),El se lucha cada mañana,NaN,el se ducha cada mañana 9,el se lucha cada mañana
4,5,¿Qué dice usted que va a hacer hoy? (9),¿Qué [gibberish] que vas estoy?,NaN,qué dice usted que va a hacer hoy 9,qué que vas estoy


In [44]:
def word_overlap(s1, s2):
    set1 = set(s1.split())
    set2 = set(s2.split())
    if len(set1) == 0:
        return 0
    return len(set1 & set2) / len(set1)

In [45]:
def score(stimulus,response):
    overlap=word_overlap(stimulus,response)
    if(overlap>0.9):
        return 4
    elif(overlap>0.75):
        return 3
    elif(overlap>0.5):
        return 2
    elif(overlap>0.3):
        return 1
    else:
        return 0
        

In [46]:
df["Score"] = df.apply(
    lambda row: score(row["clean_stimulus"], row["clean_original"]), axis=1
)
df.head(50)

,Sentence,Stimulus,Transcription Rater 1,Score,clean_stimulus,clean_original
0,1,Quiero cortarme el pelo (7),Quiero cortarme el pelo,3,quiero cortarme el pelo 7,quiero cortarme el pelo
1,2,El libro está en la mesa (7),El libro está en la mesa,3,el libro está en la mesa 7,el libro está en la mesa
2,3,El carro lo tiene Pedro (8),El carro lo tiene Pedro,3,el carro lo tiene pedro 8,el carro lo tiene pedro
3,4,El se ducha cada mañana (9),El se ducha cada mañana,3,el se ducha cada mañana 9,el se ducha cada mañana
4,5,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?,1,qué dice usted que va a hacer hoy 9,que dices ustedes se que van a hacer hoy
5,6,Dudo que sepa manejar muy bien (10),Dudo que sepa manajar bien,2,dudo que sepa manejar muy bien 10,dudo que sepa manajar bien
6,7,Las calles de esta ciudad son muy anchas (11),Las calles de esta cuidad son muy anchas,3,las calles de esta ciudad son muy anchas 11,las calles de esta cuidad son muy anchas
7,8,Puede que llueva mañana todo el día (12),Puede que lleva mañana todo el día,2,puede que llueva mañana todo el día 12,puede que lleva mañana todo el día
8,9,Las casas son muy bonitas pero caras (12),Las casas son muy bonitas pero muy cadas,2,las casas son muy bonitas pero caras 12,las casas son muy bonitas pero muy cadas
9,10,Me gustan las películas que acaban bien (12),Me gustan las peliculas que acaban bien,2,me gustan las películas que acaban bien 12,me gustan las peliculas que acaban bien


In [18]:
df.drop("Predicted_Score", axis=1, inplace=True)

In [47]:
df.head(30)

,Sentence,Stimulus,Transcription Rater 1,Score,clean_stimulus,clean_original
0,1,Quiero cortarme el pelo (7),Quiero cortarme el pelo,3,quiero cortarme el pelo 7,quiero cortarme el pelo
1,2,El libro está en la mesa (7),El libro está en la mesa,3,el libro está en la mesa 7,el libro está en la mesa
2,3,El carro lo tiene Pedro (8),El carro lo tiene Pedro,3,el carro lo tiene pedro 8,el carro lo tiene pedro
3,4,El se ducha cada mañana (9),El se ducha cada mañana,3,el se ducha cada mañana 9,el se ducha cada mañana
4,5,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?,1,qué dice usted que va a hacer hoy 9,que dices ustedes se que van a hacer hoy
5,6,Dudo que sepa manejar muy bien (10),Dudo que sepa manajar bien,2,dudo que sepa manejar muy bien 10,dudo que sepa manajar bien
6,7,Las calles de esta ciudad son muy anchas (11),Las calles de esta cuidad son muy anchas,3,las calles de esta ciudad son muy anchas 11,las calles de esta cuidad son muy anchas
7,8,Puede que llueva mañana todo el día (12),Puede que lleva mañana todo el día,2,puede que llueva mañana todo el día 12,puede que lleva mañana todo el día
8,9,Las casas son muy bonitas pero caras (12),Las casas son muy bonitas pero muy cadas,2,las casas son muy bonitas pero caras 12,las casas son muy bonitas pero muy cadas
9,10,Me gustan las películas que acaban bien (12),Me gustan las peliculas que acaban bien,2,me gustan las películas que acaban bien 12,me gustan las peliculas que acaban bien


In [22]:
df.to_excel("basic_score.xlsx", index=False)

### Rule Based+ML(Semantic) based Scoring

In [26]:
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import sys
!"{sys.executable}" -m pip install sentence-transformers

  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.3.0-py3-none-any.whl (512 kB)



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Users\Shruti Sachan\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [48]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [49]:
model=SentenceTransformer('all-MiniLM-L6-v2')

C:\Users\Shruti Sachan\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [50]:
def semantic(s1,s2):
    emb1=model.encode(s1)
    emb2=model.encode(s2)
    sim=cosine_similarity([emb1],[emb2])[0][0]
    return sim

In [51]:
from difflib import SequenceMatcher
def edit_sim(s1,s2):
    return SequenceMatcher(None,s1,s2).ratio()

In [52]:
def missing_word(s1,s2):
    word1=set(s1.split())
    word2=set(s2.split())
    missing=len(word1-word2)
    return missing/len(word1)

In [53]:
def get_ngrams(text, n):
    words = text.split()
    return [" ".join(words[i:i+n]) for i in range(len(words)-n+1)]

In [54]:
def ngram_overlap(s1, s2, n=2):
    ngrams1 = set(get_ngrams(s1, n))
    ngrams2 = set(get_ngrams(s2, n))
    if len(ngrams1) == 0:
        return 0
    return len(ngrams1 & ngrams2) / len(ngrams1)

In [55]:
def length_ratio(s1, s2):
    if len(s1.split()) == 0:
        return 0
    return len(s2.split()) / len(s1.split())

In [56]:
def hybrid_score(stimulus, response):
    overlap = word_overlap(stimulus, response)
    semantic_sim = semantic(stimulus, response)
    length = length_ratio(stimulus, response)
    match = edit_sim(stimulus, response)
    bigram_score = ngram_overlap(stimulus, response, n=2)
    final = (
        0.2 * overlap +
        0.25 * semantic_sim +
        0.15 * length +
        0.2 * match +
        0.2 * bigram_score
    )
    if length < 0.6:
        final *= 0.75
    if missing_word(stimulus, response) > 0.4:
        final *= 0.75
    if semantic_sim > 0.9 and overlap > 0.85:
        return 4
    if final > 0.85:
        return 4
    elif final > 0.7:
        return 3
    elif final > 0.5:
        return 2
    elif final > 0.3:
        return 1
    else:
        return 0

In [57]:
df["Hybrid_Score"] = df.apply(
    lambda row: hybrid_score(row["clean_stimulus"], row["clean_original"]), axis=1
)

In [58]:
df[["Stimulus", "Transcription Rater 1", "Score", "Hybrid_Score"]].head(10)

,Stimulus,Transcription Rater 1,Score,Hybrid_Score
0,Quiero cortarme el pelo (7),Quiero cortarme el pelo,3,4
1,El libro está en la mesa (7),El libro está en la mesa,3,4
2,El carro lo tiene Pedro (8),El carro lo tiene Pedro,3,4
3,El se ducha cada mañana (9),El se ducha cada mañana,3,4
4,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?,1,1
5,Dudo que sepa manejar muy bien (10),Dudo que sepa manajar bien,2,1
6,Las calles de esta ciudad son muy anchas (11),Las calles de esta cuidad son muy anchas,3,3
7,Puede que llueva mañana todo el día (12),Puede que lleva mañana todo el día,2,3
8,Las casas son muy bonitas pero caras (12),Las casas son muy bonitas pero muy cadas,2,3
9,Me gustan las películas que acaban bien (12),Me gustan las peliculas que acaban bien,2,3


In [59]:
df.head(30)

,Sentence,Stimulus,Transcription Rater 1,Score,clean_stimulus,clean_original,Hybrid_Score
0,1,Quiero cortarme el pelo (7),Quiero cortarme el pelo,3,quiero cortarme el pelo 7,quiero cortarme el pelo,4
1,2,El libro está en la mesa (7),El libro está en la mesa,3,el libro está en la mesa 7,el libro está en la mesa,4
2,3,El carro lo tiene Pedro (8),El carro lo tiene Pedro,3,el carro lo tiene pedro 8,el carro lo tiene pedro,4
3,4,El se ducha cada mañana (9),El se ducha cada mañana,3,el se ducha cada mañana 9,el se ducha cada mañana,4
4,5,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?,1,qué dice usted que va a hacer hoy 9,que dices ustedes se que van a hacer hoy,1
5,6,Dudo que sepa manejar muy bien (10),Dudo que sepa manajar bien,2,dudo que sepa manejar muy bien 10,dudo que sepa manajar bien,1
6,7,Las calles de esta ciudad son muy anchas (11),Las calles de esta cuidad son muy anchas,3,las calles de esta ciudad son muy anchas 11,las calles de esta cuidad son muy anchas,3
7,8,Puede que llueva mañana todo el día (12),Puede que lleva mañana todo el día,2,puede que llueva mañana todo el día 12,puede que lleva mañana todo el día,3
8,9,Las casas son muy bonitas pero caras (12),Las casas son muy bonitas pero muy cadas,2,las casas son muy bonitas pero caras 12,las casas son muy bonitas pero muy cadas,3
9,10,Me gustan las películas que acaban bien (12),Me gustan las peliculas que acaban bien,2,me gustan las películas que acaban bien 12,me gustan las peliculas que acaban bien,3


In [32]:
!pip install xlsxwriter

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import sys
!"{sys.executable}" -m pip install xlsxwriter

  Using cached xlsxwriter-3.2.9-py3-none-any.whl.metadata (2.7 kB)
Using cached xlsxwriter-3.2.9-py3-none-any.whl (175 kB)



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Users\Shruti Sachan\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [60]:
print(df.columns)

Index(['Sentence', 'Stimulus', 'Transcription Rater 1', 'Score',
       'clean_stimulus', 'clean_original', 'Hybrid_Score'],
      dtype='object')


In [75]:
import pandas as pd
xls = pd.ExcelFile("AutoEIT Sample Transcriptions for Scoring.xlsx")
with pd.ExcelWriter("final_output.xlsx") as writer:
    for sheet in xls.sheet_names[1:]:
        df = pd.read_excel(xls, sheet_name=sheet) 
        df["clean_stimulus"]=df["Stimulus"].apply(preprocess)
        df["clean_original"]=df["Transcription Rater 1"].apply(preprocess)
        df["Score"] = df.apply(
            lambda row: hybrid_score(row["clean_stimulus"], row["clean_original"]), axis=1
        )
        df.to_excel(writer, sheet_name=sheet, index=False)